# 🎬 Limpieza de Datos — Netflix Titles
Análisis y limpieza del dataset `netflix_titles.csv` usando **PySpark**.

## 1. Iniciar SparkSession y cargar el CSV

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, isnull, trim, lower, regexp_replace, split, to_date

spark = SparkSession.builder.master("local").appName("NetflixCleaning").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
file_path = "netflix_titles-3.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True, multiLine=True, escape='"')
df.show(5, truncate=50)

## 2. Exploración inicial del DataFrame

In [ ]:
print(f"Total de registros: {df.count()}")
print(f"Total de columnas: {len(df.columns)}")
print(f"Columnas: {df.columns}")

In [ ]:
df.printSchema()

## 3. Detección de valores nulos por columna

In [ ]:
null_counts = df.select([
    count(when(isnull(c), c)).alias(c) for c in df.columns
])
null_counts.show(truncate=False)

In [ ]:
# Porcentaje de nulos por columna
total = df.count()
null_pct = df.select([
    (count(when(isnull(c), c)) / total * 100).alias(c) for c in df.columns
])
null_pct.show(truncate=False)

## 4. Conteo de registros duplicados

In [ ]:
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
print(f"Registros totales: {total_rows}")
print(f"Registros únicos: {distinct_rows}")
print(f"Duplicados encontrados: {total_rows - distinct_rows}")

## 5. Eliminar duplicados

In [ ]:
df_clean = df.dropDuplicates()
print(f"Registros después de eliminar duplicados: {df_clean.count()}")

## 6. Rellenar valores nulos
Se rellenan los campos de texto nulos con `'Unknown'` y se eliminan filas donde `title` o `type` sean nulos (columnas críticas).

In [ ]:
# Eliminar filas donde columnas críticas son nulas
df_clean = df_clean.dropna(subset=["show_id", "type", "title"])

# Rellenar nulos en columnas de texto con 'Unknown'
fill_cols = ["director", "cast", "country", "rating", "listed_in", "description"]
fill_dict = {c: "Unknown" for c in fill_cols}
df_clean = df_clean.fillna(fill_dict)

print(f"Registros después de limpiar nulos críticos: {df_clean.count()}")

## 7. Limpieza de columnas de texto
Se aplica `trim()` para eliminar espacios en blanco y se normaliza la columna `type` a minúsculas.

In [ ]:
df_clean = df_clean.withColumn("title", trim(col("title")))
df_clean = df_clean.withColumn("director", trim(col("director")))
df_clean = df_clean.withColumn("country", trim(col("country")))
df_clean = df_clean.withColumn("type", lower(trim(col("type"))))
df_clean = df_clean.withColumn("rating", trim(col("rating")))
df_clean.select("title", "type", "director", "country", "rating").show(10, truncate=40)

## 8. Convertir `date_added` a tipo fecha

In [ ]:
df_clean = df_clean.withColumn(
    "date_added",
    to_date(trim(col("date_added")), "MMMM d, yyyy")
)
df_clean.select("title", "date_added").show(10)

## 9. Conversión de `release_year` a entero

In [ ]:
df_clean = df_clean.withColumn("release_year", col("release_year").cast("int"))
df_clean.select("title", "release_year").show(10)

## 10. Análisis: distribución por tipo de contenido

In [ ]:
from pyspark.sql.functions import desc

df_clean.groupBy("type").count().orderBy(desc("count")).show()

## 11. Análisis: Top 10 países con más contenido

In [ ]:
df_clean.filter(col("country") != "Unknown") \
    .groupBy("country").count() \
    .orderBy(desc("count")) \
    .show(10)

## 12. Análisis: Contenido por año de lanzamiento

In [ ]:
df_clean.groupBy("release_year").count() \
    .orderBy(desc("release_year")) \
    .show(15)

## 13. Verificación final de nulos después de la limpieza

In [ ]:
null_final = df_clean.select([
    count(when(isnull(c), c)).alias(c) for c in df_clean.columns
])
null_final.show(truncate=False)

## 14. Guardar el DataFrame limpio como CSV

In [ ]:
df_clean.coalesce(1).write.csv("netflix_clean_output", header=True, mode="overwrite")
print("✅ Dataset limpio guardado en: netflix_clean_output/")

## ✅ Resumen del proceso de limpieza

| Paso | Acción |
|------|--------|
| 1 | Carga del CSV con PySpark |
| 2 | Exploración: schema, filas, columnas |
| 3 | Detección y porcentaje de nulos |
| 4-5 | Eliminación de duplicados |
| 6 | Relleno de nulos y drop de filas críticas |
| 7 | Trim y normalización de texto |
| 8 | Conversión de `date_added` a fecha |
| 9 | Cast de `release_year` a entero |
| 10-12 | Análisis exploratorio post-limpieza |
| 13 | Verificación final de nulos |
| 14 | Exportación del dataset limpio |